# Estatística Descritiva
 
Objetivos: descobrir a quantidade de palavras, sentenças, adjetivos e advérbios nos relatórios nos períodos `t-1`, `t`  e `t+2`.

# Carregar as bibliotecas e o dataset

In [ ]:
import re
import pandas as pd
from unidecode import unidecode
df = pd.read_excel("D:/TCC/Codigos_e_Dastasets/Classes Gramaticais/Dataset_Original/Relatorio_Planilha_Analise_Sentimento1.xlsx")

In [2]:
print(df.head(3))

   Id_Empresa       Nome                                                t-1  \
0           1  Petrobras  A Petróleo Brasileiro S.A.  Petrobras dedicase...   
1           2      Ambev  A Companhia de Bebidas das Américas – Ambev (r...   
2           3    Via S.A  \n\nA Via Varejo S.A., diretamente ou por meio...   

                                                   t  \
0  A Petróleo Brasileiro S.A. - Petrobras dedica-...   
1  \nA Ambev S.A. (referida como “Companhia” ou “...   
2  \n\nA Via Varejo S.A., diretamente ou por meio...   

                                                 t+2  
0  A Petróleo Brasileiro S.A. – Petrobras, Socied...  
1  A Ambev S.A. (referida como “Companhia”, “Ambe...  
2  A Via S.A., diretamente ou por meio de suas co...  


# Análise de palavras e sentenças com NLTK

**O que acontece aqui:**
- instala o tokenizador punkt do NLTK
- importa funções de tokenização
- define uma função chamada contar_palavras_sentencas_nltk


**Como a função funciona:**
Para cada texto:
- se o valor estiver vazio ou ausente, ela retorna (0, 0)
- caso contrário:
  - usa sent_tokenize(...) para dividir em sentenças
  - usa word_tokenize(...) para dividir em palavras
  
Depois, ela retorna:
- quantidade de palavras
- quantidade de sentenças

In [10]:
import nltk
nltk.download('punkt')

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

def contar_palavras_sentencas_nltk(df, coluna_texto="texto"):
    """
    Conta número de palavras e número de sentenças usando NLTK.
    Retorna o DataFrame original com duas novas colunas:
    - n_palavras_nltk
    - n_sentencas_nltk
    """
    
    def processar(texto):
        if pd.isna(texto):
            return 0, 0
        
        # Sentenças
        sentencas = sent_tokenize(texto, language="portuguese")
        
        # Palavras
        palavras = word_tokenize(texto, language="portuguese")
        
        return len(palavras), len(sentencas)
    
    df[f"{coluna_texto}_n_palavras_nltk"], df[f"{coluna_texto}_n_sentencas_nltk"] = zip(*df[coluna_texto].apply(processar))
    return df

df_nltk = df.copy()
df_nltk = contar_palavras_sentencas_nltk(df_nltk, coluna_texto="t-1")
df_nltk = contar_palavras_sentencas_nltk(df_nltk, coluna_texto="t")
df_nltk = contar_palavras_sentencas_nltk(df_nltk, coluna_texto="t+2")

df_nltk = df_nltk.drop(columns=['Nome','t-1','t','t+2'])
df_nltk

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,Id_Empresa,t-1_n_palavras_nltk,t-1_n_sentencas_nltk,t_n_palavras_nltk,t_n_sentencas_nltk,t+2_n_palavras_nltk,t+2_n_sentencas_nltk
0,1,5336,154,5203,123,5226,144
1,2,5455,175,5492,157,5489,177
2,3,5564,153,5459,139,5379,172
3,4,5546,139,5390,141,5458,148
4,5,5445,149,5569,155,5576,141
5,6,5454,169,5486,174,5546,175
6,8,5470,143,5510,138,5323,146
7,9,5473,160,5318,159,5345,166
8,10,5459,149,5445,150,5585,163
9,11,5308,166,5316,164,5308,163


#  Análise de palavras e sentenças com spaCy
Este bloco faz praticamente a mesma coisa do bloco anterior, mas usando spaCy em vez de NLTK.

**O que muda:**
- importa spacy
- carrega o modelo pt_core_news_lg
- define uma função contar_palavras_sentencas_spacy

**Como a função funciona:**
Para cada texto:
- se estiver ausente, retorna zero para palavras e sentenças
- caso contrário:
    - cria um objeto Doc com nlp(texto)
    - conta todos os tokens com len(list(doc))
    - conta as sentenças com len(list(doc.sents))

In [20]:
import spacy

def contar_palavras_sentencas_spacy(df, coluna_texto="texto", modelo="pt_core_news_lg"):
    """
    Conta número de palavras e número de sentenças usando spaCy.
    Retorna o DataFrame original com duas novas colunas:
    - n_palavras_spacy
    - n_sentencas_spacy
    """
    
    nlp = spacy.load(modelo)
    
    palavras_list = []
    sentencas_list = []
    
    for texto in df[coluna_texto]:
        if pd.isna(texto):
            palavras_list.append(0)
            sentencas_list.append(0)
            continue
        
        doc = nlp(texto)
        
        #n_palavras = len([t for t in doc if t.is_alpha])  # contar só palavras
        #n_palavras = len([t for t in doc if t.is_space])  # Contar todos os tokens, exceto espaços
        n_palavras = len(list(doc))                       # Contar todos os tokens, até os espaços
        n_sentencas = len(list(doc.sents))
        
        palavras_list.append(n_palavras)
        sentencas_list.append(n_sentencas)
    
    df[f"{coluna_texto}_n_palavras_spacy"] = palavras_list
    df[f"{coluna_texto}_n_sentencas_spacy"] = sentencas_list
    
    return df

df_spacy1 = df.copy()
df_spacy1 = contar_palavras_sentencas_spacy(df_spacy1 , coluna_texto="t-1")
df_spacy1 = contar_palavras_sentencas_spacy(df_spacy1, coluna_texto="t")
df_spacy1 = contar_palavras_sentencas_spacy(df_spacy1, coluna_texto="t+2")

df_spacy1 = df_spacy1.drop(columns=['Nome','t-1','t','t+2'])
df_spacy1 

,Id_Empresa,t-1_n_palavras_spacy,t-1_n_sentencas_spacy,t_n_palavras_spacy,t_n_sentencas_spacy,t+2_n_palavras_spacy,t+2_n_sentencas_spacy
0,1,5466,162,5272,133,5315,152
1,2,5567,180,5594,166,5593,184
2,3,5642,183,5555,165,5513,204
3,4,5639,162,5478,149,5564,171
4,5,5551,163,5675,175,5687,166
5,6,5543,190,5573,191,5648,207
6,8,5569,156,5589,154,5403,162
7,9,5571,167,5426,171,5453,176
8,10,5599,170,5552,167,5668,183
9,11,5378,168,5385,167,5374,166


# Contagem de adjetivos e advérbios com NLTK
Este bloco vai além da contagem de palavras e sentenças. Ele tenta identificar classes gramaticais.

**O que acontece:**
- importa mac_morpho e os taggers do NLTK
- treina um tagger de sequência com o corpus mac_morpho

Isso significa que o notebook tenta "rotular" cada palavra com sua função gramatical, como:
- adjetivo
- advérbio

**Função criada:**
`contar_adj_adv_nltk(texto):`
- se o texto estiver vazio, retorna (0, 0)
- tokeniza o texto
- aplica o tagger
- conta quantas palavras foram etiquetadas como:
    - ADJ → adjetivo
    - ADV → advérbio

In [25]:
import nltk
from nltk.corpus import mac_morpho
from nltk.tag import UnigramTagger, BigramTagger

# Treinar um tagger para PT-BR com mac_morpho
tagged_sents = mac_morpho.tagged_sents()
uni_tagger = UnigramTagger(tagged_sents)
bi_tagger = BigramTagger(tagged_sents, backoff=uni_tagger)

def contar_adj_adv_nltk(texto):
    if pd.isna(texto) or not isinstance(texto, str):
        return 0, 0
    tokens = nltk.word_tokenize(texto, language="portuguese")
    tags = bi_tagger.tag(tokens)

    # etiquetas do mac_morpho:
    # ADJETIVO → "ADJ"
    # ADVÉRBIO → "ADV"
    adj = sum(1 for _, tag in tags if tag == "ADJ")
    adv = sum(1 for _, tag in tags if tag == "ADV")

    return adj, adv

df2 = df.copy()

# t-1
df2["adj_t_1"], df2["adv_t_1"] = zip(*df2["t-1"].apply(contar_adj_adv_nltk))

# t
df2["adj_t"], df2["adv_t"] = zip(*df2["t"].apply(contar_adj_adv_nltk))

# t+2
df2["adj_t2"], df2["adv_t2"] = zip(*df2["t+2"].apply(contar_adj_adv_nltk))

df2 = df2.drop(columns=['Nome','t-1','t','t+2'])
df2



,Id_Empresa,adj_t_1,adv_t_1,adj_t,adv_t,adj_t2,adv_t2
0,1,404,144,318,131,339,154
1,2,390,144,379,164,397,142
2,3,391,131,394,103,408,104
3,4,351,115,343,108,306,112
4,5,438,160,423,158,455,152
5,6,359,105,375,107,350,100
6,8,367,127,372,131,432,166
7,9,383,117,390,157,379,147
8,10,406,144,416,151,361,116
9,11,383,122,374,130,378,132


# Contagem de adjetivos e advérbios com spaCy
Este é o último bloco e é muito similar ao anterior, mas usando spaCy.

**O que acontece:**
- carrega novamente o modelo `pt_core_news_lg`
- define `contar_adj_adv_spacy(texto)`

**Como ele funciona:**
Para cada texto:
- cria um objeto `Doc`
- verifica cada token com token.pos_
- conta os tokens cujo rótulo é:
    - "ADJ" para adjetivo
    - "ADV" para advérbio

In [28]:
import spacy
import pandas as pd

nlp = spacy.load("pt_core_news_lg")

def contar_adj_adv_spacy(texto):
    if pd.isna(texto) or not isinstance(texto, str):
        return 0, 0
    
    doc = nlp(texto)

    adj = sum(1 for token in doc if token.pos_ == "ADJ")
    adv = sum(1 for token in doc if token.pos_ == "ADV")

    return adj, adv

df2 = df.copy()

# t-1
df2["adj_t_1"], df2["adv_t_1"] = zip(*df2["t-1"].apply(contar_adj_adv_spacy))

# t
df2["adj_t"], df2["adv_t"] = zip(*df2["t"].apply(contar_adj_adv_spacy))

# t+2
df2["adj_t2"], df2["adv_t2"] = zip(*df2["t+2"].apply(contar_adj_adv_spacy))

df2 = df2.drop(columns=['Nome','t-1','t','t+2'])
df2


,Id_Empresa,adj_t_1,adv_t_1,adj_t,adv_t,adj_t2,adv_t2
0,1,468,148,358,164,380,166
1,2,450,139,450,164,460,143
2,3,445,130,438,113,452,106
3,4,384,113,387,108,354,114
4,5,478,151,454,151,480,154
5,6,399,79,403,84,395,77
6,8,433,128,427,130,488,154
7,9,404,101,424,152,415,147
8,10,478,133,486,141,427,108
9,11,457,138,451,140,457,141
